# 12.2 Formatting options for display

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

The `display` command uses a few simple rules for choosing a good arrangement of
`data`. By changing several options, you can control overall arrangement, handling of zero
values, and line width. These options are summarized in [Table 12-1](./12_2_formatting_options_for_display.ipynb#`table`-12-1), with default values
shown in parentheses.

Arrangement of lists and tables
The `display` of a one-dimensional parameter or variable can produce a very long list,
as in this example from the scheduling `model` of [Figure 16-5](#missing) <!--- xTODO missing reference --->:

```ampl
ampl: display required;
required [*] :=
Fri1 100
Fri2   78
Fri3   52
Mon1 100
Mon2   78
Mon3   52
Sat1 100
Sat2   78
Thu1 100
Thu2   78
Thu3   52
Tue1 100
Tue2   78
Tue3   52
Wed1 100
Wed2   78
Wed3   52
;
```

The option display_1col can be used to request a more compact format:

```ampl
ampl: option display_1col 0;
ampl: display required;
required [*] :=
Fri1 100   Mon1 100   Sat1 100   Thu2 78   Tue2 78    Wed2    78
Fri2 78    Mon2 78    Sat2 78    Thu3 52   Tue3 52    Wed3    52
Fri3 52    Mon3 52    Thu1 100   Tue1 100  Wed1 100
;
```

The one-column list format is used when the number of values to be displayed is less than
or equal to display_1col, and the compact format is used otherwise. The default for
display_1col is 20; set it to zero to force the compact format, or to a very large number
to force the list format.

Multi-dimensional displays are affected by option display_1col in an analogous
way. The one-column list format is used when the number of values is less than or equal
to display_1col, while the appropriate compact format — in this case, a `table` — is
used otherwise. We showed an example of the difference in the previous section, where
the `display` for supply appeared as a list because it had only 9 values, while the `display`
for demand appeared as a `table` because its 21 values exceed the default setting of 20 for
option display_1col.

Since a parameter or variable indexed over a set of ordered pairs is also considered to
be two-dimensional, the value of display_1col affects its `display` as well. Here is the
`table` format for the parameter cost indexed over the set LINKS (from [Figure 6-2a](../06/6_2_subsets_and_slices_of_ordered_pairs.ipynb#fig-6-2a)) that
was displayed in the preceding section:

```ampl
ampl: option display_1col 0;
ampl: display cost;
cost [*,*] (tr)
:   CLEV GARY PITT            :=
DET    9   14    .
FRA   27    .   24
FRE    .    .   99
LAF   17    8    .
LAN   12   11    .
STL   26   16   28
WIN    9    .   13
;
```

A dot (.) entry indicates a nonexistent combination in the index set. Thus in the GARY
column of the `table`, there is a dot in the FRA row because the pair (GARY,FRA) is not a
member of LINKS; no cost["GARY","FRA"] is defined for this problem. On the
other hand, LINKS does contain the pair (GARY,LAF), and cost["GARY","LAF"]
is shown as 8 in the `table`.

In choosing an orientation for tables, the `display` command by default favors rows
over columns; that is, if the number of columns would exceed the number of rows, the
`table` is transposed. Thus the `table` for demand in the previous section has rows labeled
by the first coordinate and columns by the second, because it is indexed over DEST
with 7 members and then PROD with 3 members. By contrast, the `table` for cost has columns
labeled by the first coordinate and rows by the second, because it is indexed over ORIG
with 3 members and then DEST with 7 members. A transposed `table` is indicated by a
(tr) in its first line.
The transposition status of a `table` can be reversed by changing the
display_transpose option. Positive values tend to force transposition:

```ampl
ampl: option display_transpose 5;
ampl: display demand;
demand [*,*] (tr)
:       DET   FRA   FRE   LAF   LAN               STL    WIN      :=
bands   300   300   225   250   100               650     75
coils   750   500   850   500   400               950    250
plate   100   100   100   250     0               200     50
;
```

while negative values tend to suppress it:

```ampl
ampl: option display_transpose -5;
ampl: display cost;
cost [*,*]
:     DET FRA FRE LAF LAN STL                 WIN       :=
CLEV     9  27   .    17   12   26              9
GARY   14   .    .      8  11   16             .

PITT   .    24   99   .    .    28             13
;
```

The rule is as follows: a `table` is transposed only when the number of rows minus the
number of columns would be less than display_transpose. At its default value of
zero, display_transpose gives the previously described default behavior.

**Control of line width**

The option display_width gives the maximum number of characters on a line
generated by `display` (as seen in the `model` of [Figure 16-4](#missing) <!--- xTODO missing reference --->):

```ampl
ampl: option display_width 50, display_1col 0;
ampl: display required;
required [*] :=
Fri1 100   Mon3 52    Thu3 52    Wed2   78
Fri2 78    Sat1 100   Tue1 100   Wed3   52
Fri3 52    Sat2 78    Tue2 78
Mon1 100   Thu1 100   Tue3 52
Mon2 78    Thu2 78    Wed1 100
;
```

When a `table` would be wider than display_width, it is cut vertically into two or
more tables. The row names in each `table` are the same, but the columns are different:

```ampl
ampl: option display_width 50; display cost;
cost [*,*]
:          C118 C138 C140 C246 C250 C251 D237 D239                   :=
Coullard      6    9    8    7   11   10   4     5
Daskin       11    8    7    6    9   10   1     5
Hazen         9   10   11    1    5    6   2     7
Hopp         11    9    8   10    6    5   1     7
Iravani       3    2    8    9   10   11   1     5
Linetsky     11    9   10    5    3    4   6     7
Mehrotra      6   11   10    9    8    7   1     2
Nelson       11    5    4    6    7    8   1     9
Smilowitz    11    9   10    8    6    5   7     3
Tamhane       5    6    9    8    4    3   7    10
White        11    9    8    4    6    5   3    10

:         D241 M233 M239            :=
Coullard     3   2    1
Daskin       4   2    3
Hazen        8   3    4
Hopp         4   2    3
Iravani      4   6    7
Linetsky     8   1    2
Mehrotra     5   4    3
Nelson      10   2    3
Smilowitz    4   1    2
Tamhane     11   2    1
White        7   2    1
;
```

If a table's column headings are much wider than the values, `display` introduces
abbreviations to keep all columns together ([Figure 4-4](../04/4_3_a_model_of_production_and_transportation.ipynb#fig-4-4)):

```ampl
ampl: option display_width 40;
ampl: display {p in PROD, t in 1..T} (revenue[p,t]*Sell[p,t],
ampl?    prodcost[p]*Make[p,t], invcost[p]*Inv[p,t]);
# $1 = revenue[p,t]*Sell[p,t]
# $2 = prodcost[p]*Make[p,t]
# $3 = invcost[p]*Inv[p,t]
:           $1      $2       $3    :=
bands 1   150000   59900        0
bands 2   156000   60000        0
bands 3    37800   14000        0
bands 4    54000   20000        0
coils 1     9210   15477   3300
coils 2    87500   15400        0
coils 3   129500   38500        0
coils 4   163800   46200        0
;
```

On the other hand, where the headings are narrower than the values, you may be able to
squeeze more on a line by reducing the option gutter_width — the number of spaces
between columns — from its default value of 3 to 2 or 1.
**Suppression of zeros**

In some kinds of linear programs that have many more variables than constraints,
most of the variables have an optimal value of zero. For instance in the assignment problem
of [Figure 3-2](../03/3_3_other_formulations_of_the_transportation_problem.ipynb#fig-3-2), the optimal values of all the variables form this `table`, in which there is
a single 1 in each row and each column:

```ampl
ampl: display Trans;
Trans [*,*]
:         C118 C138 C140 C246 C250 C251 D237 D239 D241 M233 M239 :=
Coullard    1    0    0    0    0    0    0    0    0    0    0
Daskin      0    0    0    0    0    0    0    0    1    0    0
Hazen       0    0    0    1    0    0    0    0    0    0    0
Hopp        0    0    0    0    0    0    1    0    0    0    0
Iravani     0    1    0    0    0    0    0    0    0    0    0
Linetsky    0    0    0    0    1    0    0    0    0    0    0
Mehrotra    0    0    0    0    0    0    0    1    0    0    0
Nelson      0    0    1    0    0    0    0    0    0    0    0
Smilowitz   0    0    0    0    0    0    0    0    0    1    0
Tamhane     0    0    0    0    0    1    0    0    0    0    0
White       0    0    0    0    0    0    0    0    0    0    1
;
```

By setting omit_zero_rows to 1, all the zero values are suppressed, and the list
comes down to the entries of interest:

```ampl
ampl: option omit_zero_rows 1;
ampl: display Trans;
Trans :=
Coullard      C118      1
Daskin        D241      1
Hazen         C246      1
Hopp          D237      1
Iravani       C138      1
Linetsky      C250      1
Mehrotra      D239      1
Nelson        C140      1
Smilowitz     M233      1
Tamhane       C251      1
White         M239      1
;
```

If the number of nonzero entries is less than the value of display_1col, the `data` is
printed as a list, as it is here. If the number of nonzeros is greater than display_1col,
a `table` format would be used, and the omit_zero_rows option would only suppress
`table` rows that contain all zero entries.

For example, the `display` of the three-dimensional variable Trans from earlier in this
chapter would be condensed to the following:

```ampl
ampl: display Trans;
Trans [CLEV,*,*]
:   bands coils plate  :=
DET    0    750    0
LAF    0    500    0
LAN    0    400    0
STL    0     50    0
WIN    0    250    0
[GARY,*,*]
:    bands coils plate :=
FRE    225   850   100
LAF    250     0     0
STL    650   900   200
[PITT,*,*]
:    bands coils plate :=
DET    300     0   100
FRA    300   500   100
LAF      0     0   250
LAN    100     0     0
WIN     75     0    50
;
```

A corresponding option omit_zero_cols suppresses all-zero columns when set to 1,
and would eliminate two columns from Trans[CLEV,*,*].